In [ ]:
['start_xy', 'end_xy', 'start_yaw', 'end_yaw', 'start_speed', 'end_speed', 'avg_speed', 'direction', 'turn_label', 'lane_change']

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from tqdm import tqdm
import os

IMG_ROOT = Path('/zfsauton/scratch/eshau/imgs')
INSTRUCTION_ROOT = Path('/zfsauton/scratch/mineuih/waymax_rs/annotations/')
TARGET_ROOT = Path('/zfsauton/scratch/mineuih/waymax_rs/manual_instruction/')
os.makedirs(TARGET_ROOT, exist_ok=True)

for file_idx in range(1000):
    file_path = INSTRUCTION_ROOT / f"training_tfexample.tfrecord-{file_idx:05d}-of-01000_t10.jsonl"
    with open(file_path, 'r', encoding='utf-8') as f:
        records = []
        scenario_index = 0
        for i, item in tqdm(enumerate(f), desc=f"File {file_idx:05d}"):
            record = json.loads(item)
            annot = record['annotation']['ego_motion']
            if scenario_index != record['scenario_index']:
                continue
            # speed instruction
            if float(annot['start_speed']) < 0.1:
                if float(annot['end_speed']) < 0.1:
                    speed_inst = 'stop'
                elif float(annot['end_speed']) > 0.1:
                    speed_inst = 'stop and go'
            elif np.abs(float(annot['start_speed']) - float(annot['end_speed'])) / float(annot['start_speed']) < 0.1:
                speed_inst = 'maintain'
            elif float(annot['start_speed']) < float(annot['end_speed']):
                speed_inst = 'accelerate'
            else:
                speed_inst = 'decelerate'
            lane_is_known = True
            if speed_inst == 'stop':
                instruction = 'stop'
            else:
                instruction = ''
                if speed_inst == 'stop and go':
                    instruction += 'stop for a while, and then '

                if annot['direction'] == 'straight':
                    instruction += 'go straight '
                elif annot['direction'] == 'left':
                    instruction += 'go left '
                elif annot['direction'] == 'right':
                    instruction += 'go right '
                elif annot['direction'] == 'slight left':
                    instruction += 'go slightly left '
                elif annot['direction'] == 'slight right':
                    instruction += 'go slightly right '
                else:
                    instruction += 'go '

                if 'left' in annot['turn_label']:
                    instruction += 'to turn left '
                elif 'right' in annot['turn_label']:
                    instruction += 'to turn right '
                elif 'u-turn' in annot['turn_label']:
                    instruction += 'to make a U-turn '
                
                if 'left' in annot['lane_change']:
                    instruction += 'while changing lane to the left '
                elif 'right' in annot['lane_change']:
                    instruction += 'while changing lane to the right '
                elif 'none' in annot['lane_change']:
                    instruction += 'while following current lane '
                else:
                    lane_is_known = False
                
                if speed_inst == 'accelerate':
                    instruction += 'and accelerating' if lane_is_known else 'while accelerating'
                elif speed_inst == 'decelerate':
                    instruction += 'and slowing down' if lane_is_known else 'while slowing down'
            record['instruction'] = instruction
            records.append(record)

            scenario_index += 1
        with open(TARGET_ROOT / f"training_tfexample.tfrecord-{file_idx:05d}-of-01000_t10.jsonl", 'w') as f:
            for record in records:
                f.write(json.dumps(record) + '\n')


File 00000: 455it [00:00, 35691.20it/s]
File 00001: 479it [00:00, 58342.19it/s]


File 00002: 514it [00:00, 49174.80it/s]
File 00003: 479it [00:00, 41357.64it/s]
File 00004: 495it [00:00, 79419.34it/s]
File 00005: 465it [00:00, 43834.03it/s]
File 00006: 516it [00:00, 13248.33it/s]
File 00007: 468it [00:00, 36708.20it/s]
File 00008: 499it [00:00, 37855.55it/s]
File 00009: 481it [00:00, 48957.97it/s]
File 00010: 476it [00:00, 40300.54it/s]
File 00011: 501it [00:00, 41484.31it/s]
File 00012: 509it [00:00, 62805.98it/s]
File 00013: 476it [00:00, 41750.08it/s]
File 00014: 484it [00:00, 40370.75it/s]
File 00015: 494it [00:00, 43250.18it/s]
File 00016: 468it [00:00, 67040.10it/s]
File 00017: 453it [00:00, 45436.54it/s]
File 00018: 465it [00:00, 39319.22it/s]
File 00019: 487it [00:00, 77185.08it/s]
File 00020: 476it [00:00, 14114.35it/s]
File 00021: 450it [00:00, 39725.48it/s]
File 00022: 469it [00:00, 45358.99it/s]
File 00023: 478it [00:00, 42702.39it/s]
File 00024: 463it [00:00, 42009.27it/s]
File 00025: 499it [00:00, 50843.14it/s]
File 00026: 501it [00:00, 36955.85it/s]

KeyboardInterrupt: 